# Tennessee Property Values — Data Cleaning

Loads raw Zillow and Redfin files, filters to Tennessee metro areas, merges them, and exports a clean file to `data/processed/` for Power BI.

**Sources:**
- Zillow ZHVI All Homes Time Series (Smoothed, Seasonally Adjusted)
- Redfin Metro Market Tracker

**Output:** `data/processed/TN_property_values.csv`

In [2]:
import pandas as pd

## Load Raw Data

In [3]:
sales = pd.read_csv(
    '../data/raw/redfin_metro_market_tracker.tsv000 (1)',
    sep='\t',
    low_memory=False
)
sales.columns = sales.columns.str.lower()

values = pd.read_csv('../data/raw/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv')

In [4]:
sales.head()

,period_begin,period_end,period_duration,region_type,region_type_id,table_id,is_seasonally_adjusted,region,city,state,...,sold_above_list_yoy,price_drops,price_drops_mom,price_drops_yoy,off_market_in_two_weeks,off_market_in_two_weeks_mom,off_market_in_two_weeks_yoy,parent_metro_region,parent_metro_region_metro_code,last_updated
0,2022-05-01,2022-05-31,30,metro,-2,35840,False,"North Port, FL metro area",NaN,NaN,...,-0.078905,0.319149,0.235816,0.091876,0.382353,-0.205882,-0.106536,"North Port, FL",35840,2026-06-02 14:33:24.470 Z
1,2016-12-01,2016-12-31,30,metro,-2,45104,False,"Tacoma, WA metro area",NaN,NaN,...,0.041258,0.182605,-0.091795,-0.001002,0.317161,-0.015194,0.163363,"Tacoma, WA",45104,2026-06-02 14:33:24.470 Z
2,2018-09-01,2018-09-30,30,metro,-2,36340,False,"Oil City, PA metro area",NaN,NaN,...,0.130435,NaN,NaN,NaN,0.000000,0.000000,0.000000,"Oil City, PA",36340,2026-06-02 14:33:24.470 Z
3,2014-07-01,2014-07-31,30,metro,-2,10860,False,"Alice, TX metro area",NaN,NaN,...,0.250000,NaN,NaN,NaN,0.000000,0.000000,0.000000,"Alice, TX",10860,2026-06-02 14:33:24.470 Z
4,2023-10-01,2023-10-31,30,metro,-2,13720,False,"Big Stone Gap, VA metro area",NaN,NaN,...,0.185484,0.241758,0.007716,NaN,0.411765,-0.020053,0.314991,"Big Stone Gap, VA",13720,2026-06-02 14:33:24.470 Z


In [5]:
values.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31
0,102001,0,United States,country,NaN,124226.850719,124445.405574,124716.445774,125297.773801,125966.812663,...,366338.323837,366583.636859,367024.612896,367710.944113,368513.889428,369279.951949,370005.007795,370503.068810,370583.649909,370320.231928
1,394913,1,"New York, NY",msa,NY,220241.022643,221176.922725,222121.490928,224035.551610,226017.627731,...,702535.550236,703949.305049,706306.806277,709680.532631,713132.753507,716225.158400,719831.723350,723655.272546,726086.278463,727625.487581
2,753899,2,"Los Angeles, CA",msa,CA,225669.811207,226509.589232,227628.165553,229853.756673,232287.855633,...,957302.987799,958379.278078,960840.171876,964218.923597,968252.041247,970828.794664,972083.425340,971743.362813,970091.338711,968608.414008
3,394463,3,"Chicago, IL",msa,IL,156798.618056,156943.767900,157220.208333,157907.923431,158735.161975,...,341428.373236,342790.550639,344123.864996,345705.325594,347475.565448,349131.015117,350915.816595,352352.255440,353373.098703,353998.242501
4,394514,4,"Dallas, TX",msa,TX,131332.841965,131391.338072,131458.585335,131633.219413,131863.469663,...,373414.212536,372748.386147,372389.688238,372118.383227,371894.357439,371448.263585,370856.387408,369771.000631,368421.163266,366823.210136


## Clean Redfin Sales Data

Filter to Tennessee, all residential property type, and keep only the columns we need.

In [6]:
sales = sales.loc[
    (sales['property_type_id'] == -1) & (sales['state_code'] == 'TN'),
    [
        'period_end',
        'state_code',
        'parent_metro_region',
        'property_type',
        'median_sale_price',
        'median_list_price',
        'homes_sold',
        'new_listings',
        'inventory',
        'months_of_supply',
        'median_dom',
        'avg_sale_to_list',
        'sold_above_list',
        'price_drops',
        'off_market_in_two_weeks'
    ]
]

sales = sales.drop_duplicates(subset=['parent_metro_region', 'period_end'], keep='first')

print(f'Rows: {len(sales):,}')
print(f'Regions: {sales["parent_metro_region"].nunique()}')
sales.head()

Rows: 4,345
Regions: 26


,period_end,state_code,parent_metro_region,property_type,median_sale_price,median_list_price,homes_sold,new_listings,inventory,months_of_supply,median_dom,avg_sale_to_list,sold_above_list,price_drops,off_market_in_two_weeks
26,2017-01-31,TN,"Lewisburg, TN",All Residential,160900.0,165500.0,31.0,34.0,116.0,3.7,102.0,0.954814,0.290323,0.068966,0.341463
236,2025-12-31,TN,"Athens, TN",All Residential,268000.0,295235.0,68.0,67.0,247.0,3.6,87.0,0.926226,0.117647,0.198381,0.135593
518,2016-05-31,TN,"McMinnville, TN",All Residential,119000.0,109700.0,33.0,44.0,175.0,5.3,78.0,0.950720,0.181818,0.120000,0.173913
519,2013-02-28,TN,"Newport, TN",All Residential,102500.0,119900.0,9.0,11.0,145.0,16.1,242.0,0.937266,0.000000,NaN,0.000000
531,2021-08-31,TN,"Jackson, TN",All Residential,200899.0,194900.0,228.0,274.0,477.0,2.1,46.0,1.000666,0.421053,NaN,0.052863


## Clean Zillow Values Data

Filter to Tennessee, drop unused columns, and unpivot from wide to long format.

In [7]:
values = (
    values.loc[values['StateName'] == 'TN']
    .drop(columns=['RegionID', 'SizeRank', 'RegionType'])
)

values = values.melt(
    id_vars=['RegionName', 'StateName'],
    var_name='period_end',
    value_name='estimated_value'
)

print(f'Rows: {len(values):,}')
print(f'Regions: {values["RegionName"].nunique()}')
values.head()

Rows: 8,242
Regions: 26


,RegionName,StateName,period_end,estimated_value
0,"Nashville, TN",TN,2000-01-31,145090.194887
1,"Memphis, TN",TN,2000-01-31,117154.929977
2,"Knoxville, TN",TN,2000-01-31,107500.890903
3,"Chattanooga, TN",TN,2000-01-31,119548.045614
4,"Clarksville, TN",TN,2000-01-31,113481.835294


## Check Region Name Alignment

Inner join drops any region not in both datasets — verify what gets dropped.

In [8]:
redfin_regions = set(sales['parent_metro_region'].unique())
zillow_regions = set(values['RegionName'].unique())

print('In Redfin but not Zillow:', redfin_regions - zillow_regions)
print('In Zillow but not Redfin:', zillow_regions - redfin_regions)

In Redfin but not Zillow: {'Brownsville, TN'}
In Zillow but not Redfin: {'Chattanooga, TN'}


## Merge

Inner join on region name and period_end — only keeps rows present in both datasets.

In [9]:
merged = (
    pd.merge(
        left=sales,
        right=values,
        left_on=['parent_metro_region', 'period_end'],
        right_on=['RegionName', 'period_end'],
        how='inner',
        validate='one_to_one'
    )
    .drop(columns=['RegionName', 'StateName'])
)

print(f'Rows: {len(merged):,}')
print(f'Regions: {merged["parent_metro_region"].nunique()}')
merged.head()

Rows: 4,191
Regions: 25


,period_end,state_code,parent_metro_region,property_type,median_sale_price,median_list_price,homes_sold,new_listings,inventory,months_of_supply,median_dom,avg_sale_to_list,sold_above_list,price_drops,off_market_in_two_weeks,estimated_value
0,2017-01-31,TN,"Lewisburg, TN",All Residential,160900.0,165500.0,31.0,34.0,116.0,3.7,102.0,0.954814,0.290323,0.068966,0.341463,147747.111170
1,2025-12-31,TN,"Athens, TN",All Residential,268000.0,295235.0,68.0,67.0,247.0,3.6,87.0,0.926226,0.117647,0.198381,0.135593,256650.048934
2,2016-05-31,TN,"McMinnville, TN",All Residential,119000.0,109700.0,33.0,44.0,175.0,5.3,78.0,0.950720,0.181818,0.120000,0.173913,111496.649260
3,2013-02-28,TN,"Newport, TN",All Residential,102500.0,119900.0,9.0,11.0,145.0,16.1,242.0,0.937266,0.000000,NaN,0.000000,108818.554852
4,2021-08-31,TN,"Jackson, TN",All Residential,200899.0,194900.0,228.0,274.0,477.0,2.1,46.0,1.000666,0.421053,NaN,0.052863,171131.256365


## Add Date Columns

Add month, month name, and year so Power BI can use them directly for seasonality analysis.

In [10]:
merged['period_end'] = pd.to_datetime(merged['period_end'])

merged['month'] = merged['period_end'].dt.month
merged['month_name'] = merged['period_end'].dt.strftime('%b')
merged['year'] = merged['period_end'].dt.year

merged.dtypes

period_end                 datetime64[ns]
state_code                         object
parent_metro_region                object
property_type                      object
median_sale_price                 float64
median_list_price                 float64
homes_sold                        float64
new_listings                      float64
inventory                         float64
months_of_supply                  float64
median_dom                        float64
avg_sale_to_list                  float64
sold_above_list                   float64
price_drops                       float64
off_market_in_two_weeks           float64
estimated_value                   float64
month                               int32
month_name                         object
year                                int32
dtype: object

## Verify Date Range per Region

In [11]:
merged.groupby('parent_metro_region')['period_end'].agg(['min', 'max'])

,min,max
parent_metro_region,,
"Athens, TN",2012-01-31,2026-05-31
"Clarksville, TN",2012-07-31,2026-05-31
"Cleveland, TN",2012-01-31,2026-05-31
"Cookeville, TN",2012-01-31,2026-05-31
"Crossville, TN",2012-01-31,2026-05-31
"Dayton, TN",2012-01-31,2026-05-31
"Dyersburg, TN",2012-02-29,2026-05-31
"Greeneville, TN",2012-01-31,2026-05-31
"Jackson, TN",2012-04-30,2026-05-31


## Export to Processed

In [12]:
merged.to_csv('../data/processed/TN_property_values.csv', index=False)
print('Exported successfully.')
print(f'Shape: {merged.shape}')
merged.head()

Exported successfully.
Shape: (4191, 19)


,period_end,state_code,parent_metro_region,property_type,median_sale_price,median_list_price,homes_sold,new_listings,inventory,months_of_supply,median_dom,avg_sale_to_list,sold_above_list,price_drops,off_market_in_two_weeks,estimated_value,month,month_name,year
0,2017-01-31,TN,"Lewisburg, TN",All Residential,160900.0,165500.0,31.0,34.0,116.0,3.7,102.0,0.954814,0.290323,0.068966,0.341463,147747.111170,1,Jan,2017
1,2025-12-31,TN,"Athens, TN",All Residential,268000.0,295235.0,68.0,67.0,247.0,3.6,87.0,0.926226,0.117647,0.198381,0.135593,256650.048934,12,Dec,2025
2,2016-05-31,TN,"McMinnville, TN",All Residential,119000.0,109700.0,33.0,44.0,175.0,5.3,78.0,0.950720,0.181818,0.120000,0.173913,111496.649260,5,May,2016
3,2013-02-28,TN,"Newport, TN",All Residential,102500.0,119900.0,9.0,11.0,145.0,16.1,242.0,0.937266,0.000000,NaN,0.000000,108818.554852,2,Feb,2013
4,2021-08-31,TN,"Jackson, TN",All Residential,200899.0,194900.0,228.0,274.0,477.0,2.1,46.0,1.000666,0.421053,NaN,0.052863,171131.256365,8,Aug,2021
